### 📈 The Only Video on Investing You Need

##### ▶️ Related Quant Guild Videos:

- [The 5 Papers That Built Modern Quant Finance](https://youtu.be/ZwS1gMGegrM)

- [I Bet You've Never Found Alpha (and I Can Prove It)](https://youtu.be/UzTJHs3-eT0)

- [Quant Ranks Retail Trading Mistakes that Blow Up Your Account](https://youtu.be/1mpNxBaBeOw)

- [Non-Stationarity and Why Market Timing Fails](https://youtu.be/7nvjrgqKjJE)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

- [How to Read Options Chains](https://youtu.be/RrRbz6oXwxE)

###### ______________________________________________________________________________________________________________________________________

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [🛡️ Learn to Run a Personal Hedge Fund](https://quantguild.com/personal-hedge-fund)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

##### 📈 Risk and Reward

##### 💊 Hard to Swallow Pills
- 🔮 Literally nobody in the world knows what will happen or what the outcome of a single trade will be
- 🥪 There is no free lunch (*you don't get something for nothing*), to generate returns you must assume risk
- 🏋 There does not exist a silver bullet strategy, just like there isn't a one size fits all gym routine
- 🕒 Insane wealth is built incredibly slowly and over time, if you want lottery ticket wealth this isn't the field for you

###### ______________________________________________________________________________________________________________________________________

All of the below are considered *risky assets* in a portfolio...

 🏛️ **Examples of Securities:**
 - Stocks (equities)
 - Bonds
 - ETFs (Exchange-Traded Funds)
 - Mutual Funds
 - Options
 - Futures Contracts
 - REITs (Real Estate Investment Trusts)
 
 🖼️ **Examples of Non-Securities:**
 - Real Estate (physical property)
 - Commodities (gold, oil, agricultural products)
 - Art and Collectibles
 - Private Businesses (ownership in private companies)
 - Cryptocurrencies (Bitcoin, Ethereum, etc.)
 - Cash and Savings Accounts
 - Insurance Products (annuities, whole life policies)

 We measure returns using the expectation or *average* and risk as deviation from that expectation or *standard deviation*
 
 $$\mathbb{E}[r] = \frac{1}{n} \sum_{i=1}^n r_i$$
 $$\sigma = \sqrt{\frac{1}{n}\sum_{i=1}^n \left(r_i - \mathbb{E}[r]\right)^2}$$

 **Problem:** We can estimate these going backward, but that doesn't have to mean anything for the future.  In fact, we can estimate these perfectly for the future and still get unlucky.  

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

SEED = 3
rng = np.random.default_rng(SEED)

TRADING_DAYS_PER_YEAR = 252
PAST_YEARS = 1
FORWARD_YEARS = 1
PAST_DAYS = PAST_YEARS * TRADING_DAYS_PER_YEAR
FORWARD_DAYS = FORWARD_YEARS * TRADING_DAYS_PER_YEAR
DT = 1 / TRADING_DAYS_PER_YEAR

INITIAL_VALUE = 100.0
PAST_START_DATE = "2025-01-02"
N_FORWARD_PATHS = 30

TRUE_MU = 0.1
TRUE_SIGMA = 0.22

FRAME_STRIDE = 5
FRAME_DURATION = 20
INITIAL_I = 5

OUTPUT_HTML = "past_vs_forward_gbm_animation.html"
SHOW_FIG = True

# ============================================================
# GBM helpers
# ============================================================

def simulate_gbm_path(s0, mu, sigma, n_days, rng):
    z = rng.normal(size=n_days)
    log_returns = (
        (mu - 0.5 * sigma**2) * DT
        + sigma * np.sqrt(DT) * z
    )
    return s0 * np.exp(np.r_[0.0, np.cumsum(log_returns)])


def gbm_moments(s0, mu, sigma, times):
    """Analytical level expectation and variance for GBM."""
    mean = s0 * np.exp(mu * times)
    variance = (
        s0**2
        * np.exp(2 * mu * times)
        * (np.exp(sigma**2 * times) - 1)
    )
    return mean, variance


def estimate_gbm_parameters(path):
    """MLE-style annualized estimates from observed log returns."""
    log_returns = np.diff(np.log(path))
    sigma_hat = log_returns.std(ddof=1) / np.sqrt(DT)
    mean_log_return = log_returns.mean() / DT
    mu_hat = mean_log_return + 0.5 * sigma_hat**2
    return float(mu_hat), float(sigma_hat)


def padded_range(values, pad_fraction=0.08, min_pad=1.0):
    values = np.asarray(values, dtype=float)
    v_min = float(np.nanmin(values))
    v_max = float(np.nanmax(values))
    if np.isclose(v_min, v_max):
        pad = max(abs(v_max) * pad_fraction, min_pad)
    else:
        pad = max((v_max - v_min) * pad_fraction, min_pad)
    return [max(0, v_min - pad), v_max + pad]

# ============================================================
# Simulate one past realization and estimate its parameters
# ============================================================

past_dates = pd.bdate_range(start=PAST_START_DATE, periods=PAST_DAYS + 1)
past_times = np.arange(PAST_DAYS + 1) * DT
past_path = simulate_gbm_path(
    INITIAL_VALUE,
    TRUE_MU,
    TRUE_SIGMA,
    PAST_DAYS,
    rng,
)

mu_hat, sigma_hat = estimate_gbm_parameters(past_path)

# Past expectation and variance use the same fitted parameterization that
# will be used for the forward simulations.
past_expectation, past_variance = gbm_moments(
    INITIAL_VALUE,
    mu_hat,
    sigma_hat,
    past_times,
)
past_sd = np.sqrt(past_variance)
past_lower = np.maximum(0.0, past_expectation - past_sd)
past_upper = past_expectation + past_sd

# ============================================================
# Simulate independent forward paths from the terminal past value
# ============================================================

forward_start_value = past_path[-1]
forward_dates = pd.bdate_range(
    start=past_dates[-1] + pd.offsets.BDay(1),
    periods=FORWARD_DAYS + 1,
)
forward_times = np.arange(FORWARD_DAYS + 1) * DT

forward_paths = np.column_stack([
    simulate_gbm_path(
        forward_start_value,
        mu_hat,
        sigma_hat,
        FORWARD_DAYS,
        rng,
    )
    for _ in range(N_FORWARD_PATHS)
])

# Analytical mean path, not the sample average, so it is stable and exact.
forward_mean, forward_variance = gbm_moments(
    forward_start_value,
    mu_hat,
    sigma_hat,
    forward_times,
)

# ============================================================
# Styling — matched to the supplied Plotly animation
# ============================================================

off_white = "#e0e0e0"
realized_color = "#00d4ff"
expectation_color = "#ffaa33"
forward_color_over = "#18d618"  # green
forward_color_under = "#ff3030" # red
mean_color = "#00ff88"
variance_fill = "rgba(255,170,51,0.18)"
baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.08,
    subplot_titles=("Past Performance", "Forward Performance"),
)

initial_past_end = min(INITIAL_I, PAST_DAYS)
initial_forward_end = min(INITIAL_I, FORWARD_DAYS)

# Left: variance band lower boundary (hidden line)
fig.add_trace(
    go.Scatter(
        x=past_dates[: initial_past_end + 1],
        y=past_lower[: initial_past_end + 1],
        mode="lines",
        line=dict(width=0),
        hoverinfo="skip",
        showlegend=False,
        legendgroup="past-distribution",
    ),
    row=1,
    col=1,
)

# Left: variance band upper boundary with fill to lower boundary
fig.add_trace(
    go.Scatter(
        x=past_dates[: initial_past_end + 1],
        y=past_upper[: initial_past_end + 1],
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor=variance_fill,
        name="±1 SD from GBM variance",
        legendgroup="past-distribution",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Upper variance band: %{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

# Left: expectation
fig.add_trace(
    go.Scatter(
        x=past_dates[: initial_past_end + 1],
        y=past_expectation[: initial_past_end + 1],
        mode="lines",
        line=dict(color=expectation_color, width=3, dash="dash"),
        name="Expectation",
        legendgroup="past-distribution",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Expected value: %{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

# Left: one realized past path
fig.add_trace(
    go.Scatter(
        x=past_dates[: initial_past_end + 1],
        y=past_path[: initial_past_end + 1],
        mode="lines",
        line=dict(color=realized_color, width=3),
        name="Observed past path",
        legendgroup="past-path",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Observed value: %{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

# Right: 30 independent forward paths, color-coded by above/below expectation
# At the initial frame, compare each path's last shown value to the mean path's value at that same time
for j in range(N_FORWARD_PATHS):
    # Determine up to which point to plot for the initial frame
    yvals = forward_paths[: initial_forward_end + 1, j]
    ymean = forward_mean[: initial_forward_end + 1]
    # Use the last available value to determine color
    if yvals[-1] > ymean[-1]:
        color = forward_color_over
    else:
        color = forward_color_under
    fig.add_trace(
        go.Scatter(
            x=forward_dates[: initial_forward_end + 1],
            y=yvals,
            mode="lines",
            line=dict(color=color, width=1.5),
            opacity=0.5,
            name="30 forward simulations" if j == 0 else f"Simulation {j + 1}",
            legendgroup="forward-simulations",
            showlegend=(j == 0),
            hovertemplate=(
                f"Simulation {j + 1}<br>"
                "Date: %{x|%Y-%m-%d}<br>Value: %{y:.2f}<extra></extra>"
            ),
        ),
        row=1,
        col=2,
    )

# Right: analytical mean path, solid and fully opaque
mean_trace_index = len(fig.data)
fig.add_trace(
    go.Scatter(
        x=forward_dates[: initial_forward_end + 1],
        y=forward_mean[: initial_forward_end + 1],
        mode="lines",
        line=dict(color=mean_color, width=4),
        opacity=1.0,  # explicitly fully opaque
        name="Forward mean path",
        legendgroup="forward-mean",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Mean value: %{y:.2f}<extra></extra>",
    ),
    row=1,
    col=2,
)

# Start-value reference on the right panel
fig.add_hline(
    y=forward_start_value,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=2,
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []
max_days = max(PAST_DAYS, FORWARD_DAYS)
frame_indices = list(range(INITIAL_I, max_days + 1, FRAME_STRIDE))
if frame_indices[-1] != max_days:
    frame_indices.append(max_days)

for i in frame_indices:
    past_i = min(i, PAST_DAYS)
    forward_i = min(i, FORWARD_DAYS)
    frame_name = f"f{i}"

    frame_data = [
        go.Scatter(
            x=past_dates[: past_i + 1],
            y=past_lower[: past_i + 1],
        ),
        go.Scatter(
            x=past_dates[: past_i + 1],
            y=past_upper[: past_i + 1],
        ),
        go.Scatter(
            x=past_dates[: past_i + 1],
            y=past_expectation[: past_i + 1],
        ),
        go.Scatter(
            x=past_dates[: past_i + 1],
            y=past_path[: past_i + 1],
        ),
    ]

    # Add forward simulation paths, colored green if above current mean, red otherwise
    for j in range(N_FORWARD_PATHS):
        yvals = forward_paths[: forward_i + 1, j]
        ymean = forward_mean[: forward_i + 1]
        if len(yvals) > 0 and yvals[-1] > ymean[-1]:
            color = forward_color_over
        else:
            color = forward_color_under
        frame_data.append(
            go.Scatter(
                x=forward_dates[: forward_i + 1],
                y=yvals,
                opacity=0.5,
                line=dict(color=color, width=1.5),
                mode="lines"
            )
        )

    # Add mean path, fully opaque
    frame_data.append(
        go.Scatter(
            x=forward_dates[: forward_i + 1],
            y=forward_mean[: forward_i + 1],
            opacity=1.0,
        )
    )

    trace_indices = list(range(4 + N_FORWARD_PATHS + 1))
    frames.append(
        go.Frame(
            data=frame_data,
            traces=trace_indices,
            name=frame_name,
        )
    )

    elapsed_years = i / TRADING_DAYS_PER_YEAR
    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True,
            },
        ],
        "label": f"{elapsed_years:.1f}Y",
        "method": "animate",
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

all_values = np.r_[
    past_path,
    past_lower,
    past_upper,
    forward_paths.ravel(),
    forward_mean,
]
common_y_range = padded_range(all_values, pad_fraction=0.08)

fig.update_layout(
    title=dict(
        text="Past Performance Is One Realization — Not a Forecast",
        x=0.5,
        font=dict(color=off_white),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=670,
    width=1200,
    margin=dict(t=140, b=150, r=50, l=75),
    legend=dict(
        orientation="v",
        x=0,
        y=1,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.75)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white),
        traceorder="normal",
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": FRAME_DURATION, "redraw": False},
                        "transition": {"duration": 0},
                        "fromcurrent": True,
                    },
                ],
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "fromcurrent": True,
                    },
                ],
            },
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top",
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right",
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps,
    }],
    annotations=[
        dict(
            x=0.245,
            y=1.10,
            xref="paper",
            yref="paper",
            text=(
                "One observed path versus its fitted expectation "
                "and ±1 SD variance band"
            ),
            showarrow=False,
            font=dict(color=off_white, size=12),
        ),
        dict(
            x=0.755,
            y=1.10,
            xref="paper",
            yref="paper",
            text=(
                "Same μ and σ; 30 independent future shock sequences"
            ),
            showarrow=False,
            font=dict(color=off_white, size=12),
        ),
        dict(
            x=0.5,
            y=-0.18,
            xref="paper",
            yref="paper",
            text=(
                f"Estimated from the past path: μ = {mu_hat:.1%} annually, "
                f"σ = {sigma_hat:.1%} annually. "
            ),
            showarrow=False,
            font=dict(color=off_white, size=13),
            align="center",
        ),
    ],
)

fig.update_annotations(font=dict(color=off_white))

fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[past_dates[0], past_dates[-1]],
    title_text="Past date",
)
fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=common_y_range,
    title_text="Index value",
)

fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[forward_dates[0], forward_dates[-1]],
    title_text="Forward date",
)
fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=common_y_range,
    title_text="Index value",
)

# ============================================================
# Save / show
# ============================================================
fig.show()

###### ______________________________________________________________________________________________________________________________________

##### 🌊 Not all risk is equivalent

Just because something is more volatility, it doesn not mean that it has to or will outpreform...

###### ______________________________________________________________________________________________________________________________________

##### 🐉 Math makes big swings dangerous in the long run

Volatility induces a drag on your portfolio returns, chasing big returns in the short run can hurt long run growth...

$$(1 + r_{\text{avg}}) = \exp\left(\mu - \frac{1}{2}\sigma^2\right)$$

---

#### 📉 Efficient Market Hypothesis

 The Efficient Market Hypothesis (EMH) comes in three classical forms:
 
 1. **Weak Form EMH**: 
    - All current asset prices fully reflect all past trading data (such as historical prices and volume).
    - Technical analysis cannot produce excess returns.
 
 2. **Semi-Strong Form EMH**: 
    - All publicly available information (including fundamental data, news, and past trading data) is reflected in prices.
    - Neither technical nor fundamental analysis can consistently generate excess returns.
 
 3. **Strong Form EMH**: 
    - All information, public and private (including insider information), is fully reflected in prices.
    - No one can consistently achieve excess returns, even with access to insider information.
 
 The forms represent increasing levels of market efficiency.

Even the most developed markets can't predict the future, academia ruins an entire generation of practitioners who blindly follow this hypothesis.

If there weren't opportunity in the market, price wouldnt move.

###### ______________________________________________________________________________________________________________________________________

#### ⚖️ What is Equilibrium Price (*current market prices*)

Supply and demand create the equilibrium price on exchanges, even the price an illiquid asset may trade at albeit infrequently (houses, watches, etc...)

It represents the market's *best guess* as to what the asset is to do in the future compressing all potential risk and return to come in the future.

It is not necessarily a good representation of what the asset is worth now, or in the future.

###### ______________________________________________________________________________________________________________________________________

#### 💰 Without Counterfactuals, Trading is Poker at Best

We can *never* discern whether the outcome of a trade or investment is due to a specific signal, or even our thesis.

Academically, our thesis can be "correct" and another driver can move the market in the direction our thesis "predicts".

Our goal isn't to take big swings, betting the house at risk of losing our money, rather to be the casino and accumulate wealth patiently over time.

**If that sounds boring, welcome, it is - that's how real funds make money**

---

#### 📊 Portfolio Allocations & Diversification

The typical investor is concerned with exchange traded securities, but we will cover why non-securities may also be useful.

Securities you may hold in your portfolio include:
 - Stocks (equities)
 - Bonds
 - ETFs (Exchange-Traded Funds)
 - Mutual Funds
 - Options
 - Futures Contracts
 - REITs (Real Estate Investment Trusts)

Let's look at a few different portfolios consisting of one stock, tech stocks, and 30 stocks across different sectors...

###### ______________________________________________________________________________________________________________________________________

#### 🚩 Undiversifiable Risk and Market Beta

We see the benefits of diversification above, but in a crisis that benefit goes away.  Why?

I have an entire [quantitative research note written on this idea available for free on Quant Guild](https://quantguild.com/personal-hedge-fund)

When market risk occurs, correlations between assets goes to one together...

###### ______________________________________________________________________________________________________________________________________

#### 🎲 Physical Decorrelation

So how can we diversify undiversifiable risk?  We need to hold then different *markets* in our portfolio...

In other words, this is where non-securities and alts may be of use...

$$\text{Portfolio Variance} = w_1^2 \sigma_1^2 + w_2^2 \sigma_2^2 + 2w_1w_2\rho\sigma_1\sigma_2$$

Not just alts, but other strategies also offer this *structural diversification* - exactly the strategy I follow and what I teach on [Quant Guild](https://quantguild.com/personal-hedge-fund)

###### ______________________________________________________________________________________________________________________________________

#### 💡 What Should I Hold in my Portfolio?

Is equivalent to "*what should I do at the gym?*" there is no **best** that anyone should follow, it depends entirely on your goals.

Because forward looking return estimation is garbage without a crystal ball, it's all about positioning and survival.

If you want to bulk, don't rip cardio everyday.  If you want to get lean, don't bulk.  I can't answer this for you.  

**Nobody knows what will happen in the future**, all we can do is go to the gym everyday, put in our reps, and make progress toward our goal.  But we can prevent unnecessary injuries and roadblocks to progress with proper strategy.

---

#### 🎯 Portfolio Performance Measures & Backtests

Performance measures are always *backward looking* and mean literally nothing for whats to come.

Nobody knows which direction the market winds will blow (*beta*) and if you are trading an "*alpha*" there is no reason why it can't dry up tomorrow.



 $$ \text{Sharpe Ratio} = \frac{\mathbb{E}[R_p - R_f]}{\sigma_p} $$

 $$\displaystyle \text{Sharpe Ratio} = \frac{\mathbb{E}[R_p - R_f]}{\sigma_p}$$

 $$\displaystyle \text{MDD} = \max_{t \in [0,T]} \left( \frac{\text{Peak}_t - \text{Trough}_t}{\text{Peak}_t} \right )$$

 $$\displaystyle \text{CAGR} = \left( \frac{\text{Ending Value}}{\text{Beginning Value}} \right)^{\frac{1}{n}} - 1$$

###### ______________________________________________________________________________________________________________________________________

#### 🔎 How Should we Think About Backtests?

The goal is not prediction, it's to assess our exposures for when the market winds blow in certain directions.

The market is not stable, it will constantly throw different regimes at us.  Periods of low, med, and high volatility.  Inflationary and deflationary periods.

We are looking to position ourselves to survive to the long run, not make ephemeral predictions and chase short run returns.

---

#### 💭 Closing Thoughts and Future Topics

 **📑 TL;DW Executive Summary** 
 - This notebook frames investing around the inescapable trade-off between *risk and reward*: to earn returns you must take risk, there is no free lunch and no silver-bullet strategy, and real wealth is compounded slowly rather than won on a single lucky trade. We measure reward with the expected return $\mathbb{E}[r]$ and risk with the standard deviation $\sigma$, while stressing that both are backward-looking estimates that need not hold going forward.
 - Not all volatility is rewarded, and large swings are actively *dangerous* over the long run because of volatility drag—$(1 + r_{\text{avg}}) = \exp\!\left(\mu - \tfrac{1}{2}\sigma^2\right)$—so chasing big short-run returns can erode long-run compound growth.
 - Markets are hard to beat but not omniscient: the three forms of the Efficient Market Hypothesis (weak, semi-strong, strong) describe increasing efficiency, yet no market can predict the future, and the equilibrium (current market) price is only the market's *best guess*—a compression of all future risk and return—rather than a statement of true worth.
 - Without counterfactuals, a single trade is *poker at best*: we can never cleanly attribute an outcome to our thesis, so the goal is to be the casino and accumulate wealth patiently, not to bet the house.
 - Diversification helps until it doesn't—in a crisis correlations rush toward one and undiversifiable *market beta* dominates—so genuine diversification requires holding structurally *decorrelated* markets and strategies (including alts and non-securities), where portfolio variance $w_1^2\sigma_1^2 + w_2^2\sigma_2^2 + 2w_1w_2\rho\sigma_1\sigma_2$ makes the role of $\rho$ explicit.
 - There is no universally *best* portfolio—like a gym routine, the right allocation depends entirely on your goals—and performance measures such as the Sharpe ratio, maximum drawdown, and CAGR are strictly backward-looking. The key message: the purpose of modeling and backtesting is **not** prediction but *positioning and survival*—we assess our exposures across regimes so that no single state of the world can wipe us out.

###### ______________________________________________________________________________________________________________________________________

 
**Future Topics**

Technical Videos and Other Discussions

 - Fama-French / Carhart and Factor Modeling in General
 - Hawkes Processes
 - Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
 - Market-Making Models and Simulation (Stoikov-Avellaneda)
 - My First Year as a Quant
 - Why Hedge Funds are Actually Secretive
 - Non-Markovian Models (fractional Brownian motion, Volterra Process)
 - Top 3 Uses of Linear Algebra for Quant Finance
 - Girsanov's Change of Measure
 - Rough Path Theory, Applications of Path Signatures
 - Sig-Vol Model, Calibration, and Pricing
 - Trading with Alternative Data Sources
 - Pairs Trading and Statistical Arbitrage
 - Data Cleaning & Outlier Handling in Financial Time Series
 - Practical Issues in Multi-Asset Portfolio Backtesting
 - Risk Premia Harvesting: Equity, FX, Rates

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How Interactive Broker's API Works (EWrapper/EClient)
- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System

---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$